# 06 - Cleaning Auxiliary Tables (Iterasi 2)

Audit kualitas data untuk 6 tabel tambahan (bureau, bureau_balance,
previous_application, POS_CASH_balance, credit_card_balance,
installments_payments) sebelum dipakai di Feature Engineering v2.

Cek yang dilakukan: profiling missing value, duplikat baris, integritas
referensial antar tabel, sentinel value (365243), nilai AMT_* negatif
tidak logis, dan konsistensi tipe data.

#  Import

In [ ]:
import sys
from pathlib import Path
from datetime import datetime
from datetime import timezone

sys.path.insert(0, str(Path.cwd().parent / "src"))

from modeling.train import load_featured_dataset
from cleaning_auxiliary import (
    load_all_auxiliary_tables, run_quality_check_all_tables, save_quality_report,
    fix_sentinel_values,save_cleaning_decision_log,save_cleaning_decision_log,TABLE_CONFIGS,
)
import config

## Load Keenam Tabel Tambahan + Daftar SK_ID_CURR Valid

`application_train_ids` dipakai sebagai acuan integritas referensial --
semua SK_ID_CURR di tabel tambahan seharusnya ada di sini.

In [18]:
df_main = load_featured_dataset()  # atau pd.read_csv ke application_train.csv langsung
application_train_ids = set(df_main[config.ID_COLUMN].unique())

auxiliary_tables = load_all_auxiliary_tables()

for name, df in auxiliary_tables.items():
    print(f"{name}: {df.shape}")

bureau: (1048575, 17)
bureau_balance: (27299925, 3)
previous_application: (1048575, 37)
POS_CASH_balance: (10001358, 8)
credit_card_balance: (3840312, 23)
installments_payments: (13605401, 8)


## Jalankan Audit Kualitas Data untuk Semua Tabel

Satu fungsi generik dipanggil sekali, meng-loop keenam tabel secara internal.

In [19]:
all_reports = run_quality_check_all_tables(auxiliary_tables, application_train_ids)

for table_name, report in all_reports.items():
    print(f"\n=== {table_name} ===")
    print("Profil       :", report["profile"]["n_rows"], "baris,", report["profile"]["n_duplicate_rows"], "duplikat")
    print("Integritas ID:", report["referential_integrity"])
    print("Sentinel 365243 ditemukan di:", report["sentinel_values"])
    print("Nilai negatif tidak logis di:", report["illogical_negative_amounts"])
    print("Masalah dtype:", report["dtype_problems"])


=== bureau ===
Profil       : 1048575 baris, 0 duplikat
Integritas ID: [{'child_id_column': 'SK_ID_CURR', 'parent_table': 'application_train', 'n_unique_child_ids': 218292, 'n_orphan_ids': 29614, 'pct_orphan': 13.5662}]
Sentinel 365243 ditemukan di: {}
Nilai negatif tidak logis di: {'AMT_CREDIT_SUM_DEBT': {'count': 4715, 'pct': 0.4497}, 'AMT_CREDIT_SUM_LIMIT': {'count': 197, 'pct': 0.0188}}
Masalah dtype: {}

=== bureau_balance ===
Profil       : 27299925 baris, 0 duplikat
Integritas ID: [{'child_id_column': 'SK_ID_BUREAU', 'parent_table': 'bureau', 'n_unique_child_ids': 817395, 'n_orphan_ids': 314363, 'pct_orphan': 38.4591}]
Sentinel 365243 ditemukan di: {}
Nilai negatif tidak logis di: {}
Masalah dtype: {}

=== previous_application ===
Profil       : 1048575 baris, 0 duplikat
Integritas ID: [{'child_id_column': 'SK_ID_CURR', 'parent_table': 'application_train', 'n_unique_child_ids': 305828, 'n_orphan_ids': 43497, 'pct_orphan': 14.2227}]
Sentinel 365243 ditemukan di: {'DAYS_FIRST_DRA

## Simpan Laporan Ringkas

Hasil audit disimpan ke `reports/auxiliary_data_quality_report.csv` --
jadi rujukan sebelum mengambil keputusan pembersihan di Feature Engineering v2.

In [20]:
report_path = save_quality_report(all_reports)
print("Laporan tersimpan di:", report_path)

import pandas as pd
pd.read_csv(report_path)

Laporan tersimpan di: D:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\reports\auxiliary_data_quality_report.csv


,table_name,n_rows,n_columns,n_duplicate_rows,n_columns_with_missing,worst_missing_column,worst_missing_pct,max_orphan_pct,n_sentinel_columns_found,n_illogical_negative_columns,n_dtype_problems
0,bureau,1048575,17,0,6,AMT_ANNUITY,69.57,13.5662,0,2,0
1,bureau_balance,27299925,3,0,0,NaN,0.00,38.4591,0,0,0
2,previous_application,1048575,37,0,15,RATE_INTEREST_PRIMARY,99.65,14.2227,5,1,0
3,POS_CASH_balance,10001358,8,0,2,CNT_INSTALMENT,0.26,39.5181,0,0,0
4,credit_card_balance,3840312,23,0,9,AMT_PAYMENT_CURRENT,20.00,44.1763,0,3,0
5,installments_payments,13605401,8,0,2,DAYS_ENTRY_PAYMENT,0.02,39.4840,0,0,0


In [21]:
import json
print(json.dumps(all_reports["previous_application"]["sentinel_values"], indent=2))
print(json.dumps(all_reports["previous_application"]["referential_integrity"], indent=2))
print(json.dumps(all_reports["bureau_balance"]["referential_integrity"], indent=2))
print(json.dumps(all_reports["POS_CASH_balance"]["referential_integrity"], indent=2))
print(json.dumps(all_reports["credit_card_balance"]["referential_integrity"], indent=2))
print(json.dumps(all_reports["installments_payments"]["referential_integrity"], indent=2))
print(json.dumps(all_reports["bureau"]["illogical_negative_amounts"], indent=2))
print(json.dumps(all_reports["previous_application"]["illogical_negative_amounts"], indent=2))
print(json.dumps(all_reports["credit_card_balance"]["illogical_negative_amounts"], indent=2))

{
  "DAYS_FIRST_DRAWING": {
    "count": 588688,
    "pct": 56.1417
  },
  "DAYS_FIRST_DUE": {
    "count": 25604,
    "pct": 2.4418
  },
  "DAYS_LAST_DUE_1ST_VERSION": {
    "count": 58840,
    "pct": 5.6114
  },
  "DAYS_LAST_DUE": {
    "count": 133012,
    "pct": 12.685
  },
  "DAYS_TERMINATION": {
    "count": 142237,
    "pct": 13.5648
  }
}
[
  {
    "child_id_column": "SK_ID_CURR",
    "parent_table": "application_train",
    "n_unique_child_ids": 305828,
    "n_orphan_ids": 43497,
    "pct_orphan": 14.2227
  }
]
[
  {
    "child_id_column": "SK_ID_BUREAU",
    "parent_table": "bureau",
    "n_unique_child_ids": 817395,
    "n_orphan_ids": 314363,
    "pct_orphan": 38.4591
  }
]
[
  {
    "child_id_column": "SK_ID_CURR",
    "parent_table": "application_train",
    "n_unique_child_ids": 337252,
    "n_orphan_ids": 47808,
    "pct_orphan": 14.1757
  },
  {
    "child_id_column": "SK_ID_PREV",
    "parent_table": "previous_application",
    "n_unique_child_ids": 936325,
    "n_orp

In [22]:
print("Dtype SK_ID_BUREAU di bureau        :", auxiliary_tables["bureau"]["SK_ID_BUREAU"].dtype)
print("Dtype SK_ID_BUREAU di bureau_balance:", auxiliary_tables["bureau_balance"]["SK_ID_BUREAU"].dtype)
print("Dtype SK_ID_PREV di previous_application  :", auxiliary_tables["previous_application"]["SK_ID_PREV"].dtype)
print("Dtype SK_ID_PREV di POS_CASH_balance       :", auxiliary_tables["POS_CASH_balance"]["SK_ID_PREV"].dtype)

bureau_ids = set(auxiliary_tables["bureau"]["SK_ID_BUREAU"].dropna().unique())
balance_ids = set(auxiliary_tables["bureau_balance"]["SK_ID_BUREAU"].dropna().unique())
orphan_sample = list(balance_ids - bureau_ids)[:10]
print("\nContoh 10 SK_ID_BUREAU yang dianggap orphan:", orphan_sample)
print("Apakah ID itu benar-benar tidak ada di bureau.csv (cek manual)?")
print(auxiliary_tables["bureau"][auxiliary_tables["bureau"]["SK_ID_BUREAU"].isin(orphan_sample)])

Dtype SK_ID_BUREAU di bureau        : int64
Dtype SK_ID_BUREAU di bureau_balance: int64
Dtype SK_ID_PREV di previous_application  : int64
Dtype SK_ID_PREV di POS_CASH_balance       : int64

Contoh 10 SK_ID_BUREAU yang dianggap orphan: [np.int64(6291456), np.int64(5242880), np.int64(5242881), np.int64(5242882), np.int64(6291462), np.int64(6291477), np.int64(5939765), np.int64(6291484), np.int64(5939766), np.int64(5242913)]
Apakah ID itu benar-benar tidak ada di bureau.csv (cek manual)?
Empty DataFrame
Columns: [SK_ID_CURR, SK_ID_BUREAU, CREDIT_ACTIVE, CREDIT_CURRENCY, DAYS_CREDIT, CREDIT_DAY_OVERDUE, DAYS_CREDIT_ENDDATE, DAYS_ENDDATE_FACT, AMT_CREDIT_MAX_OVERDUE, CNT_CREDIT_PROLONG, AMT_CREDIT_SUM, AMT_CREDIT_SUM_DEBT, AMT_CREDIT_SUM_LIMIT, AMT_CREDIT_SUM_OVERDUE, CREDIT_TYPE, DAYS_CREDIT_UPDATE, AMT_ANNUITY]
Index: []


## Terapkan Perbaikan Sentinel Value & Simpan Catatan Keputusan

Menutup tahap Cleaning Auxiliary: mengganti nilai sentinel 365243 di 5 kolom
`previous_application` jadi NaN, lalu mendokumentasikan seluruh keputusan
audit (termasuk yang TIDAK memerlukan tindakan) ke satu file JSON sebagai
bukti proses untuk portofolio.

# Terapkan Fix Sentinel

In [23]:
SENTINEL_COLUMNS_PREVIOUS_APPLICATION = [
    "DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE", "DAYS_TERMINATION",
]

auxiliary_tables["previous_application"] = fix_sentinel_values(
    auxiliary_tables["previous_application"], SENTINEL_COLUMNS_PREVIOUS_APPLICATION
)

print("Jumlah sentinel 365243 tersisa setelah fix (harus 0 semua):")
for col in SENTINEL_COLUMNS_PREVIOUS_APPLICATION:
    print(f"  {col}: {(auxiliary_tables['previous_application'][col] == 365243).sum()}")

Jumlah sentinel 365243 tersisa setelah fix (harus 0 semua):
  DAYS_FIRST_DRAWING: 0
  DAYS_FIRST_DUE: 0
  DAYS_LAST_DUE_1ST_VERSION: 0
  DAYS_LAST_DUE: 0
  DAYS_TERMINATION: 0


# Simpan Decision Log Lengkap
Ini rangkuman semua keputusan dari audit kita, termasuk yang tidak memerlukan tindakan (penting untuk didokumentasikan supaya jelas itu keputusan sadar, bukan terlewat).

In [ ]:


decision_log = {
    "stage": "cleaning_auxiliary_tables",
    "tables_audited": list(TABLE_CONFIGS.keys()),
    "decisions": [
        {
            "table": "previous_application",
            "issue": "Sentinel value 365243 di 5 kolom DAYS_*",
            "columns_affected": SENTINEL_COLUMNS_PREVIOUS_APPLICATION,
            "action": "Diganti menjadi NaN",
            "rationale": "DAYS_FIRST_DRAWING sampai 56.14% berisi nilai boneka; dibiarkan akan merusak agregasi mean/max.",
        },
        {
            "table": "previous_application",
            "issue": "RATE_INTEREST_PRIMARY hilang 99.65%",
            "columns_affected": ["RATE_INTEREST_PRIMARY"],
            "action": "Kolom di-drop, tidak dibawa ke Feature Engineering v2",
            "rationale": "Sinyal nyaris tidak ada (<0.5% terisi), konsisten dengan temuan publik di kompetisi ini.",
        },
        {
            "table": "bureau, previous_application, credit_card_balance",
            "issue": "Nilai negatif pada beberapa kolom AMT_*",
            "columns_affected": [
                "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_LIMIT", "AMT_DOWN_PAYMENT",
                "AMT_BALANCE", "AMT_DRAWINGS_ATM_CURRENT", "AMT_DRAWINGS_CURRENT",
            ],
            "action": "Tidak ada tindakan",
            "rationale": "Persentase kejadian di bawah 0.5%, kemungkinan besar representasi kelebihan bayar (valid secara bisnis), dampak ke agregasi dapat diabaikan.",
        },
        {
            "table": "bureau, previous_application",
            "issue": "Orphan SK_ID_CURR 14-16% (tidak ditemukan di application_train)",
            "columns_affected": ["SK_ID_CURR"],
            "action": "Tidak difilter manual",
            "rationale": "Sesuai proporsi client di application_test.csv (~13.68%); otomatis tersaring saat merge ke application_train di Feature Engineering v2.",
        },
        {
            "table": "bureau_balance, POS_CASH_balance, credit_card_balance, installments_payments",
            "issue": "Orphan SK_ID_BUREAU/SK_ID_PREV 38-44% (tidak ditemukan di tabel induk)",
            "columns_affected": ["SK_ID_BUREAU", "SK_ID_PREV"],
            "action": "Tidak difilter manual",
            "rationale": "Dikonfirmasi bukan bug (dtype cocok int64, verifikasi manual membuktikan ID memang tidak ada di tabel induk). Baris ini tidak bisa ditelusuri ke SK_ID_CURR mana pun sehingga otomatis tersaring saat merge berjenjang.",
        },
        {
            "table": "semua (6 tabel)",
            "issue": "Duplikat baris & konsistensi dtype",
            "columns_affected": [],
            "action": "Tidak ada tindakan",
            "rationale": "0 duplikat dan 0 masalah dtype ditemukan di seluruh tabel.",
        },
    ],
    "created_at": datetime.now(timezone.utc).isoformat(),
}

log_path = save_cleaning_decision_log(decision_log)
print("Decision log tersimpan di:", log_path)

Decision log tersimpan di: D:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\data\processed\auxiliary_cleaning_decision_log.json
